# heom_map — HEOM for the enhanced algorithm

HEOM needs no map file: the generator $\hat{\mathcal L}_{\rm HEOM}$ is time independent, so **one**
matrix exponential
$$P=e^{\hat{\mathcal L}_{\rm HEOM}\Delta t}$$
on the extended (system $\otimes$ ADO) space is the entire classical cost. That $P$ is the
"compute it once" object of the enhanced algorithm.

$P$ itself is **not** what gets re-applied on the quantum grid — it acts on the 2640-dimensional ADO
hierarchy, not on a density matrix. Instead, $P$ is used once more to hand the grid something it can
carry: `heom_shorttime_maps` propagates the $d^2$ basis elements through $P$ for $K$ steps, giving the
reduced maps $\mathcal L(\Delta t)..\mathcal L(K\Delta t)$ (qHEOM, Batista *et al.*, JCTC 2025,
arXiv:2411.12049, Eq. 33: the reduced map is the zeroth-ADO block of $e^{\hat{\mathcal L}_{\rm HEOM}t}$).
`grid.ipynb` turns those into the transfer tensors and the single fixed companion operator that the
enhanced algorithm then re-applies — exactly as for the path integral. So HEOM goes through the same
enhanced grid as every other method here; only the way its one operator is obtained differs.

**Provides:**
- `heom_onestep_propagator(...)` → $P$, n_ados (the whole classical cost)
- `heom_shorttime_maps(P, K)` → $\mathcal L(0..K\Delta t)$ from $P$ alone, for the grid
- `standard_heom_rho(...)` → populations from a direct qutip `HEOMSolver.run` (the benchmark)

In [ ]:
import numpy as np
import qutip as qt
from scipy.linalg import expm
from qutip.solver.heom import HEOMSolver, DrudeLorentzPadeBath

In [ ]:
def _make_solver(H, lam, gamma, T_K, KB_CM, Nk, depth):
    d = H.shape[0]
    baths = [DrudeLorentzPadeBath(qt.basis(d, i) * qt.basis(d, i).dag(), lam=lam, gamma=gamma, T=KB_CM * T_K, Nk=Nk)
             for i in range(d)]
    return HEOMSolver(qt.Qobj(H), baths, max_depth=depth)

In [ ]:
def heom_onestep_propagator(dt_fs, *, H, lam, gamma, T_K, KB_CM, FS_TO_CM,
                            Nk=1, depth=3, verbose=True):
    """ONE-step propagator P = exp(L_HEOM * dt) on the extended system+ADO
    space, computed once. Iterating it and reading the zeroth-ADO block
    (the first d^2 entries, with the ADOs initialised to zero) is exact for
    arbitrarily long times. Returns (P, n_ados)."""
    d = H.shape[0]
    solver = _make_solver(H, lam, gamma, T_K, KB_CM, Nk, depth)

    if verbose:
        print(f"  one-step P: {solver._n_ados} ADOs, "
              f"{solver._n_ados * d * d} x {solver._n_ados * d * d} "
              f"(one-time matrix exponential)...", flush=True)

    # Holen der vollen Superoperator-Matrix aus dem Solver
    A = solver.rhs(0).full()

    # Direktes Berechnen von exp(A * dt) mittels Padé-Approximation
    dt_cm = dt_fs * FS_TO_CM
    P = expm(A * dt_cm)

    return P, solver._n_ados

In [ ]:
def heom_shorttime_maps(P, K, d):
    """The reduced maps L(0), L(dt), ..., L(K dt) extracted from the ONE
    propagator P -- no further HEOM solve, just K matrix-vector products.

    Column j of L(m dt) is the zeroth-ADO block reached after m applications
    of P from the initial condition (system = basis element E_j, all ADOs
    zero), which is exactly qHEOM Eq. 33 evaluated at t = m dt.

    These K maps are what grid.ipynb turns into the transfer tensors and the
    one fixed companion operator that the enhanced algorithm re-applies."""
    D = d * d
    X = np.zeros((P.shape[0], D), dtype=complex)
    X[:D, :] = np.eye(D)                       # ADOs start at zero
    maps = [np.eye(D, dtype=complex)]
    for _ in range(K):
        X = P @ X
        maps.append(X[:D, :].copy())    # Wir hängen nur die ersten D Zeilen (=rho_S) an, die restlichen Zeilen enthalten ADOs
    return np.array(maps)

## heom_shorttime_maps
Maps ist eine Liste die $\rho_S(i\cdot \Delta t)$ für $i \in \{1,...,K\}$ enthält. Die $i$-te Spalte in maps gibt die Dynamik für initial excitation auf site $i$ an.

$$\text{maps} = \Big[ \text{maps}[0], \, \text{maps}[1], \, \dots, \, \text{maps}[K] \Big]$$
$$\text{maps}[0] =  \begin{pmatrix} 1 & 0 & 0 & 0 \\ 0 & 1 & 0 & 0 \\ 0 & 0 & 1 & 0 \\ 0 & 0 & 0 & 1 \end{pmatrix}$$
$$\text{maps}[1] =  \begin{pmatrix} \rho_{11}(\Delta t)_{[start=11]} & \rho_{11}(\Delta t)_{[start=12]} & \rho_{11}(\Delta t)_{[start=21]} & \rho_{11}(\Delta t)_{[start=22]} \\ \rho_{12}(\Delta t)_{[start=11]} & \rho_{12}(\Delta t)_{[start=12]} & \rho_{12}(\Delta t)_{[start=21]} & \rho_{12}(\Delta t)_{[start=22]} \\ \rho_{21}(\Delta t)_{[start=11]} & \rho_{21}(\Delta t)_{[start=12]} & \rho_{21}(\Delta t)_{[start=21]} & \rho_{21}(\Delta t)_{[start=22]} \\ \rho_{22}(\Delta t)_{[start=11]} & \rho_{22}(\Delta t)_{[start=12]} & \rho_{22}(\Delta t)_{[start=21]} & \rho_{22}(\Delta t)_{[start=22]} \end{pmatrix}$$
$$\text{maps}[K] =  \begin{pmatrix} \rho_{11}(K\Delta t)_{[start=11]} & \rho_{11}(K\Delta t)_{[start=12]} & \rho_{11}(K\Delta t)_{[start=21]} & \rho_{11}(K\Delta t)_{[start=22]} \\ \rho_{12}(K\Delta t)_{[start=11]} & \rho_{12}(K\Delta t)_{[start=12]} & \rho_{12}(K\Delta t)_{[start=21]} & \rho_{12}(K\Delta t)_{[start=22]} \\ \rho_{21}(K\Delta t)_{[start=11]} & \rho_{21}(K\Delta t)_{[start=12]} & \rho_{21}(K\Delta t)_{[start=21]} & \rho_{21}(K\Delta t)_{[start=22]} \\ \rho_{22}(K\Delta t)_{[start=11]} & \rho_{22}(K\Delta t)_{[start=12]} & \rho_{22}(K\Delta t)_{[start=21]} & \rho_{22}(K\Delta t)_{[start=22]} \end{pmatrix}$$

Das funktioniert weil jede Spalte in $$X_0 =  \begin{pmatrix} e_1 & e_2 & \dots & e_D \\ 0 & 0 & \dots & 0 \\ \vdots & \vdots & \ddots & \vdots \\ 0 & 0 & \dots & 0 \end{pmatrix} = \begin{pmatrix} \mathbb{I}_{D \times D} \\ 0 \end{pmatrix}$$
unabhängig auf den Propagator $P$ wirkt. Die $i$-te spalte von $X$ gibt die Systemdynamik und die ADOs in vektorisierter Form an:
$$X(i) =  \begin{pmatrix} \vert{}\rho_{\text{sys}}(i)\rangle\!\rangle \\ \vert{}ADO_1(i)\rangle\!\rangle \\ \vert{}ADO_2(i)\rangle\!\rangle \\ \vdots \end{pmatrix}$$
Der Propagator $P$ ist die Große HEOM matrix aus dem HEOM_theory.ipynb notebook.

In [ ]:
def standard_heom_rho(t_fs, rho0, *, H, lam, gamma, T_K, KB_CM, FS_TO_CM,
                       Nk=1, depth=3):
    """BENCHMARK: dynamics from a direct qutip HEOMSolver.run -- the ordinary
    implementation, no grid, no Kraus operators. Returns the FULL density
    matrices (nt, d, d) so that populations AND coherences can be compared."""
    solver = _make_solver(H, lam, gamma, T_K, KB_CM, Nk, depth)
    states = solver.run(qt.Qobj(np.asarray(rho0, complex)), np.asarray(t_fs) * FS_TO_CM).states
    return np.array([np.asarray(s.full()) for s in states])